# 🦀 Day 7: Mini-Project — Shape Calculator

Trait-based polymorphism with Circle, Rectangle, and Triangle!

---

## 📐 Step 1: Define the Shape Trait

In [ ]:
use std::fmt;

trait Shape: fmt::Display {
    fn area(&self) -> f64;
    fn perimeter(&self) -> f64;
    fn name(&self) -> &str;

    fn is_larger_than(&self, other: &dyn Shape) -> bool {
        self.area() > other.area()
    }

    fn scale(&self, factor: f64) -> Box<dyn Shape>;

    fn info(&self) {
        println!("📐 {}", self.name());
        println!("   {}", self);
        println!("   Area:      {:.4}", self.area());
        println!("   Perimeter: {:.4}", self.perimeter());
    }
}

println!("Shape trait defined! ✅");

## 🔴 Step 2: Circle

In [ ]:
#[derive(Debug, Clone)]
struct Circle {
    radius: f64,
}

impl Circle {
    fn new(radius: f64) -> Self {
        Circle { radius: radius.abs() }
    }

    fn diameter(&self) -> f64 {
        self.radius * 2.0
    }
}

impl Shape for Circle {
    fn area(&self) -> f64 {
        std::f64::consts::PI * self.radius.powi(2)
    }

    fn perimeter(&self) -> f64 {
        2.0 * std::f64::consts::PI * self.radius
    }

    fn name(&self) -> &str { "Circle" }

    fn scale(&self, factor: f64) -> Box<dyn Shape> {
        Box::new(Circle::new(self.radius * factor))
    }
}

impl fmt::Display for Circle {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        write!(f, "Circle(r={:.2})", self.radius)
    }
}

Circle::new(5.0).info();

## 🟦 Step 3: Rectangle

In [ ]:
#[derive(Debug, Clone)]
struct Rectangle {
    width: f64,
    height: f64,
}

impl Rectangle {
    fn new(width: f64, height: f64) -> Self {
        Rectangle { width: width.abs(), height: height.abs() }
    }

    fn square(side: f64) -> Self {
        Rectangle::new(side, side)
    }

    fn is_square(&self) -> bool {
        (self.width - self.height).abs() < f64::EPSILON
    }
}

impl Shape for Rectangle {
    fn area(&self) -> f64 { self.width * self.height }
    fn perimeter(&self) -> f64 { 2.0 * (self.width + self.height) }
    fn name(&self) -> &str { if self.is_square() { "Square" } else { "Rectangle" } }

    fn scale(&self, factor: f64) -> Box<dyn Shape> {
        Box::new(Rectangle::new(self.width * factor, self.height * factor))
    }
}

impl fmt::Display for Rectangle {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        write!(f, "Rectangle({}×{})", self.width, self.height)
    }
}

Rectangle::new(4.0, 6.0).info();
println!();
Rectangle::square(5.0).info();

## 🔺 Step 4: Triangle

In [ ]:
#[derive(Debug, Clone)]
struct Triangle {
    a: f64, b: f64, c: f64,
}

impl Triangle {
    fn new(a: f64, b: f64, c: f64) -> Option<Self> {
        let (a, b, c) = (a.abs(), b.abs(), c.abs());
        if a + b > c && b + c > a && a + c > b {
            Some(Triangle { a, b, c })
        } else {
            None
        }
    }

    fn equilateral(side: f64) -> Self {
        Triangle { a: side, b: side, c: side }
    }
}

impl Shape for Triangle {
    fn area(&self) -> f64 {
        let s = self.perimeter() / 2.0;
        (s * (s - self.a) * (s - self.b) * (s - self.c)).sqrt()
    }

    fn perimeter(&self) -> f64 { self.a + self.b + self.c }
    fn name(&self) -> &str { "Triangle" }

    fn scale(&self, factor: f64) -> Box<dyn Shape> {
        Box::new(Triangle { a: self.a * factor, b: self.b * factor, c: self.c * factor })
    }
}

impl fmt::Display for Triangle {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        write!(f, "Triangle({:.2}, {:.2}, {:.2})", self.a, self.b, self.c)
    }
}

Triangle::equilateral(6.0).info();

## 🧮 Step 5: Shape Collection

In [ ]:
struct ShapeCollection {
    shapes: Vec<Box<dyn Shape>>,
}

impl ShapeCollection {
    fn new() -> Self { ShapeCollection { shapes: Vec::new() } }

    fn add(&mut self, shape: Box<dyn Shape>) {
        self.shapes.push(shape);
    }

    fn total_area(&self) -> f64 {
        self.shapes.iter().map(|s| s.area()).sum()
    }

    fn total_perimeter(&self) -> f64 {
        self.shapes.iter().map(|s| s.perimeter()).sum()
    }

    fn largest(&self) -> Option<&dyn Shape> {
        self.shapes.iter()
            .max_by(|a, b| a.area().partial_cmp(&b.area()).unwrap())
            .map(|s| s.as_ref())
    }

    fn smallest(&self) -> Option<&dyn Shape> {
        self.shapes.iter()
            .min_by(|a, b| a.area().partial_cmp(&b.area()).unwrap())
            .map(|s| s.as_ref())
    }

    fn report(&self) {
        println!("╔══════════════════════════════════════════╗");
        println!("║         📐 SHAPE REPORT                 ║");
        println!("╠══════════════════════════════════════════╣");
        for shape in &self.shapes {
            println!("║  {:<30} A={:.2}", format!("{}", shape), shape.area());
        }
        println!("╠══════════════════════════════════════════╣");
        println!("║  Total shapes: {:<25}", self.shapes.len());
        println!("║  Total area: {:<27.2}", self.total_area());
        println!("║  Total perimeter: {:<22.2}", self.total_perimeter());
        if let Some(largest) = self.largest() {
            println!("║  Largest: {:<30}", format!("{}", largest));
        }
        println!("╚══════════════════════════════════════════╝");
    }
}

let mut collection = ShapeCollection::new();
collection.add(Box::new(Circle::new(5.0)));
collection.add(Box::new(Rectangle::new(4.0, 6.0)));
collection.add(Box::new(Triangle::equilateral(8.0)));
collection.add(Box::new(Rectangle::square(3.0)));
collection.add(Box::new(Circle::new(2.0)));

collection.report();

In [ ]:
// Scaling shapes
let circle = Circle::new(3.0);
println!("Original: {} (area={:.2})", circle, circle.area());
let scaled = circle.scale(2.0);
println!("Scaled 2x: {} (area={:.2})", scaled, scaled.area());

## 🏋️ Extension Challenges

In [ ]:
// Challenge 1: Add a Pentagon and Hexagon
// Challenge 2: Add a method to check if one shape fits inside another
// Challenge 3: Implement PartialOrd for Box<dyn Shape> based on area

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **Traits = interfaces**: define shared behavior for different types
2. **`Box<dyn Shape>`**: trait objects for heterogeneous collections
3. **Default methods**: reduce boilerplate (`is_larger_than`, `info`)
4. **Supertrait** (`Shape: Display`): shapes must be displayable
5. **Factory methods** (`scale`) return `Box<dyn Shape>` for polymorphism

---
**Next Week:** Error Handling Mastery! ⚠️